[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S17_ml_fundamentos.ipynb)

# Sesión 17 · Fundamentos de Machine Learning

**Módulo 5: Machine Learning** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Explicar qué es el aprendizaje supervisado y distinguir un problema de regresión de uno de clasificación.
2. Separar variables (`X`) y objetivo (`y`), y dividir los datos en entrenamiento y prueba con `train_test_split`.
3. Entrenar y usar un modelo de scikit-learn con `fit` y `predict`, empezando por la regresión lineal.
4. Comparar el modelo con un baseline usando el error absoluto medio (MAE) y el R².

## 📋 Qué debes saber antes
Módulos 2 a 4: NumPy, pandas y Matplotlib.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Los verificadores recalculan todo **con tu propia división** de los datos, así que tus números pueden diferir de los de un compañero y estar bien.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos del módulo 4 y carga las funciones que revisan tus respuestas. scikit-learn ya viene instalado en Colab; los imports de cada herramienta los harás tú, para aprender de dónde sale cada una.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica: ventas diarias de una tienda ----------
_n = 300
_pub = np.round(rng.uniform(0, 5000, _n), 2)
_web = np.clip(rng.normal(1200, 300, _n), 200, None).round().astype(int)
_fer = (rng.random(_n) < 0.1).astype(int)
_precio = np.round(rng.uniform(35, 60, _n), 2)
tiendas = pd.DataFrame({
    "gasto_publicidad": _pub, "visitas_web": _web, "es_feriado": _fer, "precio_promedio": _precio,
    "ventas": np.round(8000 + 1.8 * _pub + 4.5 * _web + 6000 * _fer - 90 * _precio + rng.normal(0, 1500, _n), 2),
})

# ---------- Datos de práctica: clientes de un banco ----------
_m = 400
_ing = np.round(np.clip(rng.normal(4500, 1500, _m), 1200, None), 2)
_edad = rng.integers(20, 70, _m)
_prod = rng.integers(1, 6, _m)
clientes = pd.DataFrame({
    "ingreso": _ing, "edad": _edad, "n_productos": _prod,
    "gasto_mensual": np.round(300 + 0.45 * _ing + 8 * _edad + 150 * _prod + rng.normal(0, 400, _m), 2),
})
nuevo_cliente = pd.DataFrame({"ingreso": [5000.0], "edad": [35], "n_productos": [2]})

_D = copy.deepcopy({"tiendas": tiendas, "clientes": clientes})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _ols(X, y):
    """Mínimos cuadrados con NumPy (sin scikit-learn): devuelve (coeficientes, intercepto)."""
    A = np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)])
    sol = np.linalg.lstsq(A, np.asarray(y, dtype=float), rcond=None)[0]
    return sol[1:], sol[0]


def _mae(real, pred):
    real, pred = list(map(float, real)), list(map(float, pred))
    return math.fsum(abs(a - b) for a, b in zip(real, pred)) / len(real)


def _r2(real, pred):
    real, pred = list(map(float, real)), list(map(float, pred))
    media = math.fsum(real) / len(real)
    return 1 - math.fsum((a - b) ** 2 for a, b in zip(real, pred)) / math.fsum((a - media) ** 2 for a in real)


def _particion(r, nombres, datos_X, datos_y, prop_test, columnas):
    """Revisa X_train, X_test, y_train, y_test: tipos, tamaños, sin repetir filas y alineados."""
    xs, xt, ys, yt = (r.var(n) for n in nombres)
    if any(v is _FALTA for v in (xs, xt, ys, yt)):
        return False
    if not all(isinstance(v, pd.DataFrame) for v in (xs, xt)) or not all(isinstance(v, pd.Series) for v in (ys, yt)):
        r.mal("Las X deberían ser DataFrames y las y, Series (usa `train_test_split` sobre `X` e `y`).")
        return False
    n = len(datos_X)
    n_test = math.ceil(n * prop_test)
    if len(xt) != n_test or len(xs) != n - n_test:
        r.mal(f"`{nombres[1]}` tiene {len(xt)} filas y `{nombres[0]}`, {len(xs)}: con test_size={prop_test} se esperaban {n_test} y {n - n_test}.")
        return False
    if set(xs.index) & set(xt.index) or set(xs.index) | set(xt.index) != set(datos_X.index):
        r.mal("Entrenamiento y prueba deberían repartirse todas las filas, sin repetir ninguna.")
        return False
    if list(xs.index) != list(ys.index) or list(xt.index) != list(yt.index):
        r.mal("Cada X debería estar alineada con su y (mismas filas en el mismo orden): divide X e y en la misma llamada.")
        return False
    if [str(c) for c in xs.columns] != columnas:
        r.mal(f"Las X deberían tener las columnas {columnas}.")
        return False
    r.ok("La partición en entrenamiento y prueba es correcta.")
    return True


def _array(r, nombre, largo):
    v = r.var(nombre)
    if v is _FALTA:
        return None
    arr = np.asarray(v, dtype=float).ravel() if isinstance(v, (np.ndarray, pd.Series, list)) else None
    if arr is None or len(arr) != largo:
        r.mal(f"`{nombre}` debería ser un array con {largo} valores.")
        return None
    return arr


COLS = ["gasto_publicidad", "visitas_web", "es_feriado"]


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    t = _D["tiendas"]
    _df(r, "X", COLS, t[COLS].values.tolist(), "solo las tres columnas pedidas, en ese orden")
    _ser(r, "y", t["ventas"].tolist(), "la columna ventas")
    _sin_cambios_df(r, "tiendas")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_abandono": "90b4229717a186edd559bfeb46d3907af7a4373d3ae691f4ec770200a6f88fee",
        "pred_ventas": "2347fe9fb58bef25c040f42e6c315ec67c8743eee9893f6c343cd2755da65073",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    t = _D["tiendas"]
    _particion(r, ["X_train", "X_test", "y_train", "y_test"], t, t["ventas"], 0.2, COLS)
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_filas_test": "f211408fc9156f75b0230ebe4099c486aff5a294b30380d90daf60cca7443df2",
    })
    r.fin()


def _ajuste_ref():
    xs, ys = globals().get("X_train"), globals().get("y_train")
    if isinstance(xs, pd.DataFrame) and isinstance(ys, pd.Series) and len(xs) == len(ys):
        return _ols(xs, ys)
    return None


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    ref = _ajuste_ref()
    m = r.var("modelo")
    if ref is None:
        r.mal("Primero resuelve el ejercicio 2: necesito `X_train` e `y_train`.")
    elif m is not _FALTA:
        if not hasattr(m, "coef_"):
            r.mal("`modelo` debería ser una regresión lineal **entrenada**: ¿llamaste a `fit` con `X_train` e `y_train`?")
        elif not _cerca_lista(np.ravel(m.coef_), ref[0], 1e-4) or abs(float(m.intercept_) - ref[1]) > 1e-3:
            r.mal("Los coeficientes de `modelo` no son los de una regresión lineal entrenada con `X_train` e `y_train`.")
        else:
            r.ok("`modelo` está entrenado con los datos de entrenamiento.")
    xt = globals().get("X_test")
    if ref is not None and isinstance(xt, pd.DataFrame):
        esperado = np.asarray(xt, dtype=float) @ ref[0] + ref[1]
        p = _array(r, "pred_test", len(xt))
        if p is not None:
            r.ok("`pred_test` tiene las predicciones para `X_test`.") if _cerca_lista(p, esperado, 1e-3) else \
                r.mal("`pred_test` debería salir de `modelo.predict(X_test)`.")
    c = r.var("coefs")
    if c is not _FALTA and ref is not None:
        if not isinstance(c, pd.Series) or [str(i) for i in c.index] != COLS:
            r.mal("`coefs` debería ser una Series con un coeficiente por columna de `X`, con los nombres de las columnas como índice.")
        elif not _cerca_lista(c.tolist(), ref[0], 1e-4):
            r.mal("Los valores de `coefs` deberían ser `modelo.coef_`.")
        else:
            r.ok("`coefs` es correcto.")
    if ref is not None:
        _esc(r, "intercepto", ref[1], "debería ser `modelo.intercept_`", tol=1e-3)
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    yt, ys = globals().get("y_test"), globals().get("y_train")
    ref = _ajuste_ref()
    if not isinstance(yt, pd.Series) or not isinstance(ys, pd.Series) or ref is None:
        r.mal("Primero resuelve los ejercicios 2 y 3.")
    else:
        media_train = math.fsum(ys.tolist()) / len(ys)
        b = _array(r, "baseline", len(yt))
        if b is not None:
            if all(abs(x - media_train) < 1e-6 for x in b):
                r.ok("`baseline` predice siempre el promedio de entrenamiento.")
            elif all(abs(x - statistics.fmean(yt.tolist())) < 1e-6 for x in b):
                r.mal("`baseline` usa el promedio de **prueba**: eso es usar información que el modelo no debería conocer. Usa el de entrenamiento.")
            else:
                r.mal("`baseline` debería repetir, para cada fila de prueba, el promedio de `y_train`.")
        pred = np.asarray(globals()["X_test"], dtype=float) @ ref[0] + ref[1]
        mb, mm = _mae(yt, [media_train] * len(yt)), _mae(yt, pred)
        _esc(r, "mae_base", mb, "el error absoluto medio del baseline en prueba", tol=1e-3)
        _esc(r, "mae_modelo", mm, "el error absoluto medio del modelo en prueba", tol=1e-3)
        _esc(r, "mejora_pct", round((1 - mm / mb) * 100, 1), "cuánto menor es el error del modelo que el del baseline, en porcentaje con 1 decimal", tol=0.051)
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_media_de": "0a439258a247f70cba6aac61e7d82cb85b1e5c426c8b5749d2372fb803260d65",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    yt, ref, xt = globals().get("y_test"), _ajuste_ref(), globals().get("X_test")
    if not isinstance(yt, pd.Series) or ref is None or not isinstance(xt, pd.DataFrame):
        r.mal("Primero resuelve los ejercicios 2 y 3.")
    else:
        pred = np.asarray(xt, dtype=float) @ ref[0] + ref[1]
        _esc(r, "r2", _r2(yt, pred), "el R² del modelo en los datos de prueba", tol=1e-6)
        ax = _grafico(r, "ax_real")
        if ax is not None:
            puntos = [c for c in ax.collections if isinstance(c, mpl.collections.PathCollection)]
            diagonal = [l for l in ax.get_lines() if len(l.get_xdata()) >= 2 and _cerca_lista(l.get_xdata(), l.get_ydata(), 1e-6)]
            if not puntos or not (_cerca_lista(puntos[0].get_offsets()[:, 0], yt.tolist(), 1e-6) and _cerca_lista(puntos[0].get_offsets()[:, 1], pred, 1e-3)):
                r.mal("`ax_real` debería tener un punto por fila de prueba: el valor real en x y la predicción en y.")
            elif not diagonal:
                r.mal("Agrega la línea diagonal (donde real = predicción) para ver qué tan lejos queda cada punto.")
            else:
                r.ok("`ax_real` compara lo real con lo predicho.")
            _rotulos(r, "ax_real", ax, None, "Ventas reales (S/)", "Ventas predichas (S/)")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_r2_negativo": "6c5fb3b25e6ba7dcf12155440e0c51b36a0492e24123968a10f8c31c304ebb6e",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    c = _D["clientes"]
    cols = ["ingreso", "edad", "n_productos"]
    if not _particion(r, ["Xb_train", "Xb_test", "yb_train", "yb_test"], c, c["gasto_mensual"], 0.25, cols):
        r.fin()
        return
    xs, xt, ys, yt = (globals()[n] for n in ["Xb_train", "Xb_test", "yb_train", "yb_test"])
    coef, b = _ols(xs, ys)
    m = r.var("modelo_b")
    if m is not _FALTA:
        ok = hasattr(m, "coef_") and _cerca_lista(np.ravel(m.coef_), coef, 1e-4)
        r.ok("`modelo_b` está entrenado con `Xb_train` e `yb_train`.") if ok else r.mal("`modelo_b` debería ser una regresión lineal entrenada con `Xb_train` e `yb_train`.")
    pred = np.asarray(xt, dtype=float) @ coef + b
    media = math.fsum(ys.tolist()) / len(ys)
    _esc(r, "mae_b", _mae(yt, pred), "el MAE del modelo en prueba", tol=1e-3)
    _esc(r, "mae_base_b", _mae(yt, [media] * len(yt)), "el MAE del baseline (promedio de entrenamiento) en prueba", tol=1e-3)
    _esc(r, "gasto_por_sol", coef[0], "el coeficiente del ingreso: cuánto sube el gasto por cada sol más de ingreso", tol=1e-5)
    _esc(r, "pred_nuevo", float((np.asarray(nuevo_cliente_ref, dtype=float) @ coef + b)[0]),
         "la predicción de `modelo_b` para `nuevo_cliente` (un número, no un array)", tol=1e-3)
    r.fin()


nuevo_cliente_ref = nuevo_cliente.copy()


def check_pro():
    r = _Revision("Nivel pro")
    t = _D["tiendas"]
    cols4 = COLS + ["precio_promedio"]
    xs, xt = globals().get("X_train"), globals().get("X_test")
    ys, yt = globals().get("y_train"), globals().get("y_test")
    if not all(isinstance(v, pd.DataFrame) for v in (xs, xt)):
        r.mal("Primero resuelve el ejercicio 2.")
    else:
        xs4, xt4 = t.loc[xs.index, cols4], t.loc[xt.index, cols4]
        coef, b = _ols(xs4, ys)
        mae4 = _mae(yt, np.asarray(xt4, dtype=float) @ coef + b)
        m = r.var("modelo_4")
        if m is not _FALTA:
            ok = hasattr(m, "coef_") and len(np.ravel(m.coef_)) == 4 and _cerca_lista(np.ravel(m.coef_), coef, 1e-4)
            r.ok("`modelo_4` usa las cuatro variables, con las mismas filas de entrenamiento.") if ok else \
                r.mal("`modelo_4` debería entrenarse con las cuatro columnas y exactamente las mismas filas de `X_train`.")
        _esc(r, "mae_4", mae4, "el MAE en prueba del modelo con cuatro variables", tol=1e-3)
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
- `tiendas`: 300 días de una tienda, con el gasto en publicidad del día, las visitas a la web, si fue feriado (1) o no (0), el precio promedio y las **ventas** del día.
- `clientes`: 400 clientes de un banco, con su ingreso, edad, cantidad de productos y **gasto mensual**. `nuevo_cliente` es un cliente que aún no tiene historial.

In [ ]:
print(tiendas.head(), "\n")
print(tiendas.describe().round(1), "\n")
print(clientes.head())

---
## 1. Aprendizaje supervisado: `X` e `y`

### 📘 Concepto
En el **aprendizaje supervisado**, el modelo aprende de ejemplos en los que ya conocemos la respuesta:
- **`X`** (las *variables* o *features*): lo que sabemos de cada caso; por ejemplo, el gasto en publicidad de un día.
- **`y`** (el *objetivo* o *target*): lo que queremos predecir; por ejemplo, las ventas de ese día.

El modelo busca una regla que convierta `X` en `y` y luego la aplica a casos nuevos, donde `y` todavía no se conoce.

Según qué es `y`, hay dos tipos de problema:

| Tipo | `y` es... | Ejemplos |
|---|---|---|
| **Regresión** | un número | ventas de mañana, gasto de un cliente, precio de una casa |
| **Clasificación** | una categoría | ¿el cliente abandonará? (sí/no), ¿es fraude?, ¿qué producto comprará? |

Por convención, `X` es un DataFrame (una fila por caso, una columna por variable) y `y` es una Series.

In [ ]:
casas_ej = pd.DataFrame({"metros": [60, 85, 120, 45], "dormitorios": [2, 3, 4, 1], "precio": [210_000, 290_000, 400_000, 160_000]})
X_ej = casas_ej[["metros", "dormitorios"]]     # lo que sabemos
y_ej = casas_ej["precio"]                      # lo que queremos predecir
print(X_ej.shape, y_ej.shape)

### ✍️ Tu turno · Ejercicio 1: definir el problema
**Parte A.** Queremos predecir las ventas del día.
1. `X`: las columnas `gasto_publicidad`, `visitas_web` y `es_feriado` de `tiendas`, en ese orden.
2. `y`: la columna `ventas`.

(Dejamos `precio_promedio` fuera a propósito: la usarás en el nivel pro.)

**Parte B.** Responde con `"regresión"` o `"clasificación"`:

| Variable | Problema |
|---|---|
| `pred_abandono` | predecir si un cliente cerrará su cuenta el próximo mes |
| `pred_ventas` | predecir cuánto venderá una tienda mañana |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Doble corchete para varias columnas (DataFrame) y corchete simple para una (Series).
</details>

<details><summary>💡 Pista 2</summary>

`X = tiendas[["gasto_publicidad", "visitas_web", "es_feriado"]]`. En la parte B, mira si la respuesta es un número o una categoría.
</details>

---
## 2. Entrenamiento y prueba: `train_test_split`

### 📘 Concepto
Un modelo que se evalúa con los mismos datos con los que aprendió saca una nota inflada: sería como tomar un examen con las mismas preguntas que se estudiaron. Por eso se separan los datos:
- **Entrenamiento** (*train*): de ahí aprende el modelo.
- **Prueba** (*test*): se guarda aparte y solo se usa al final, para medir cómo le iría con datos nuevos.

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

- `test_size=0.2` deja el 20 % para prueba.
- La división es al azar; `random_state` fija la semilla para que siempre salga igual.
- Devuelve **cuatro** objetos, en ese orden. Las filas de cada X siguen alineadas con las de su y.

In [ ]:
from sklearn.model_selection import train_test_split

a_ej, b_ej, c_ej, d_ej = train_test_split(X_ej, y_ej, test_size=0.25, random_state=0)
print(a_ej, "\n", c_ej)
print(len(a_ej), len(b_ej))

### ✍️ Tu turno · Ejercicio 2: separar los datos
**Parte A.** Divide `X` e `y` en `X_train`, `X_test`, `y_train` e `y_test`, con el 20 % para prueba y `random_state=42`. Imprime cuántas filas quedan en cada parte.

**Parte B.** Predice **sin ejecutar**: `pred_filas_test` = cuántas filas tendrá `X_test`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Importa `train_test_split` desde `sklearn.model_selection`.
</details>

<details><summary>💡 Pista 2</summary>

El orden de las cuatro variables importa: primero las dos X (train, test) y después las dos y (train, test).
</details>

---
## 3. La API de scikit-learn: `fit` y `predict`

### 📘 Concepto
Todos los modelos de scikit-learn se usan igual:

```python
from sklearn.linear_model import LinearRegression
modelo = LinearRegression()       # 1. crear
modelo.fit(X_train, y_train)      # 2. entrenar: aprende de los ejemplos
pred = modelo.predict(X_test)     # 3. predecir: aplica lo aprendido a casos nuevos
```

La **regresión lineal** predice `y` como una suma ponderada de las variables más una constante:

```
ventas ≈ intercepto + coef₁ · publicidad + coef₂ · visitas + coef₃ · feriado
```

Después de `fit`, los coeficientes quedan en `modelo.coef_` (uno por columna, en el orden de `X`) y la constante en `modelo.intercept_`. Cada coeficiente dice cuánto cambia la predicción cuando esa variable sube en 1 y las demás se mantienen igual.

In [ ]:
from sklearn.linear_model import LinearRegression

modelo_ej = LinearRegression()
modelo_ej.fit(X_ej, y_ej)
print(modelo_ej.coef_, modelo_ej.intercept_)
print(modelo_ej.predict(pd.DataFrame({"metros": [100], "dormitorios": [3]})))

### ✍️ Tu turno · Ejercicio 3: tu primer modelo
1. `modelo`: una regresión lineal entrenada con `X_train` e `y_train`.
2. `pred_test`: sus predicciones para `X_test`.
3. `coefs`: una Series con los coeficientes, con los nombres de las columnas de `X` como índice, e `intercepto`: la constante.

Mira `coefs`: ¿cuántos soles más de venta asocia el modelo a un día feriado? ¿Y a 100 visitas más a la web?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Sigue los tres pasos: crear, `fit` y `predict`.
</details>

<details><summary>💡 Pista 2</summary>

`coefs = pd.Series(modelo.coef_, index=X.columns)`.
</details>

---
## 4. ¿El modelo sirve? Baseline y MAE

### 📘 Concepto
Un modelo no se evalúa en el vacío: se compara con un **baseline**, la predicción más simple posible. En regresión, lo más simple es predecir siempre el **promedio de entrenamiento**. Si el modelo no le gana al baseline, no aporta nada.

El **MAE** (error absoluto medio) mide cuánto se equivoca en promedio, en las mismas unidades que `y`:

```
MAE = promedio de |real − predicho|
```

Un MAE de 1500 significa "en promedio, el modelo se equivoca en S/ 1500 por día". Se calcula con `mean_absolute_error(y_real, y_predicho)` de `sklearn.metrics`.

El baseline se arma solo con datos de **entrenamiento**: usar información de prueba para construir cualquier parte del modelo es hacer trampa, aunque sea sin querer.

In [ ]:
from sklearn.metrics import mean_absolute_error

real_ej = np.array([100, 150, 200])
base_ej = np.full(3, 140.0)                     # siempre la misma predicción
print(mean_absolute_error(real_ej, base_ej), np.mean(np.abs(real_ej - base_ej)))

### ✍️ Tu turno · Ejercicio 4: contra el baseline
**Parte A.**
1. `baseline`: un array con el promedio de `y_train` repetido una vez por cada fila de prueba.
2. `mae_base` y `mae_modelo`: el MAE en prueba del baseline y del modelo.
3. `mejora_pct`: cuánto menor es el error del modelo que el del baseline, en porcentaje con 1 decimal: `(1 - mae_modelo / mae_base) * 100`.

**Parte B.** Responde en `pred_media_de` con `"train"` o `"test"`: ¿de qué conjunto sale el promedio del baseline?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

`np.full(largo, valor)` crea un array con un mismo valor repetido.
</details>

<details><summary>💡 Pista 2</summary>

`baseline = np.full(len(y_test), y_train.mean())`. Los dos MAE se calculan contra `y_test`.
</details>

---
## 5. R² y el gráfico de real frente a predicho

### 📘 Concepto
El **R²** (coeficiente de determinación) dice qué parte de la variación de `y` explica el modelo, comparándolo con predecir siempre el promedio:
- **1**: predicción perfecta.
- **0**: igual que el promedio.
- **Negativo**: peor que predecir el promedio. Sí, puede pasar.

Se calcula con `r2_score(y_real, y_predicho)` o con `modelo.score(X_test, y_test)`.

Un gráfico de **real frente a predicho** muestra dónde se equivoca: si el modelo fuera perfecto, todos los puntos caerían sobre la diagonal.

In [ ]:
from sklearn.metrics import r2_score

print(r2_score([100, 150, 200], [110, 140, 205]))
print(r2_score([100, 150, 200], [150, 150, 150]))       # predecir el promedio da 0

### ✍️ Tu turno · Ejercicio 5: ¿cuánto explica?
**Parte A.**
1. `r2`: el R² del modelo en prueba.
2. `fig_real, ax_real`: un gráfico de dispersión con las ventas reales de prueba en x y las predichas en y (`s=20`, `alpha=0.6`), más una línea diagonal `GRIS` que vaya del menor al mayor valor real. Eje x `Ventas reales (S/)` y eje y `Ventas predichas (S/)`.

**Parte B.** Responde en `pred_r2_negativo` con `"sí"` o `"no"`: ¿puede un modelo tener R² negativo en prueba?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

La diagonal es una línea que pasa por `(mínimo, mínimo)` y `(máximo, máximo)`.
</details>

<details><summary>💡 Pista 2</summary>

`limites = [y_test.min(), y_test.max()]` y `ax_real.plot(limites, limites, color=GRIS)`.
</details>

---
## 🏋️ Reto final: ¿cuánto gastará un cliente?
Con `clientes`, repite el flujo completo para predecir el `gasto_mensual`:
1. `Xb` con `ingreso`, `edad` y `n_productos`, e `yb` con `gasto_mensual`.
2. `Xb_train`, `Xb_test`, `yb_train`, `yb_test`: 25 % para prueba y `random_state=7`.
3. `modelo_b`: una regresión lineal entrenada.
4. `mae_b` y `mae_base_b`: el MAE en prueba del modelo y del baseline.
5. `gasto_por_sol`: el coeficiente del ingreso, es decir, cuánto sube el gasto mensual por cada sol más de ingreso.
6. `pred_nuevo`: el gasto que el modelo predice para `nuevo_cliente`, como un número.

Explica en una celda de texto, en una línea, qué significa `gasto_por_sol` para el banco.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Es el mismo flujo de los ejercicios 1 a 4, con otros nombres.
</details>

<details><summary>💡 Pista 2</summary>

`gasto_por_sol = modelo_b.coef_[0]` (el ingreso es la primera columna). `modelo_b.predict(nuevo_cliente)` devuelve un array: toma su primer elemento.
</details>

---
## 🚀 Nivel pro (opcional): ¿más información, mejor modelo?
Entrena `modelo_4` con las mismas filas de entrenamiento pero agregando `precio_promedio` a las tres variables, y calcula `mae_4`, su MAE en prueba. Para tener exactamente las mismas filas, usa los índices de tu división: `tiendas.loc[X_train.index, columnas]`. ¿Bajó el error? ¿Por qué no conviene agregar variables sin pensar?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar qué es el aprendizaje supervisado y qué son `X` e `y`.
- [ ] Decir si un problema es de regresión o de clasificación.
- [ ] Explicar por qué se separan entrenamiento y prueba, y usar `train_test_split`.
- [ ] Entrenar un modelo con `fit`, predecir con `predict` e interpretar los coeficientes de una regresión lineal.
- [ ] Construir un baseline y explicar por qué se arma solo con datos de entrenamiento.
- [ ] Explicar el MAE en las unidades del problema y qué significa un R² de 0 o negativo.

**Próxima sesión (S18):** clasificación: regresión logística, árboles de decisión, KNN y el umbral de decisión.